# Thế hệ 2 — Numerical Weather Prediction (NWP) nguyên thuỷ

Richardson (1922) lần đầu thử giải trực tiếp **phương trình nguyên thuỷ** (primitive equations) bằng
sai phân hữu hạn — thất bại vì bước thời gian quá lớn gây mất ổn định số (không có điều kiện
Courant-Friedrichs-Lewy). Charney, Fjørtoft & von Neumann (1950) sửa bằng mô hình **barotropic**
đơn giản hoá (chỉ 1 tầng, phương trình xoáy thế vị bảo toàn) chạy được trên ENIAC.

Demo: tích hợp số **phương trình xoáy thế vị barotropic 1 chiều** (dạng đơn giản hoá của mô hình
Charney-Fjørtoft-von Neumann 1950) bằng sai phân hữu hạn — đúng kiểu toán của thế hệ này, không
dùng dữ liệu thật (vì đây là mô hình lý thuyết đơn giản hoá, không phải tái tạo khí quyển thật).

In [1]:
DATA_PATH = r"/Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc"
print("Dùng dữ liệu thật:", DATA_PATH)

Dùng dữ liệu thật: /Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc


In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Phương trình xoáy thế vị barotropic 1D đơn giản hoá: d(zeta)/dt + U d(zeta)/dx = 0 (advection tuyến tính)
# zeta = xoáy tương đối. Sai phân hữu hạn trung tâm theo không gian, Euler tiến theo thời gian.
N = 200          # số điểm lưới
L = 4.0e7        # chu vi vĩ tuyến ~ 40,000 km (mét)
dx = L / N
U = 20.0         # m/s, gió nền (vĩ hướng)
dt = 0.4 * dx / U   # điều kiện CFL: U*dt/dx < 1 để ổn định số (bài học của Richardson 1922)
nsteps = 400

x = np.linspace(0, L, N, endpoint=False)
zeta = np.sin(2 * np.pi * 3 * x / L)  # sóng Rossby mẫu, số sóng 3 quanh vĩ tuyến
zeta0 = zeta.copy()

cfl = U * dt / dx
print(f"dx={dx:.1f} m, dt={dt:.1f} s, CFL = U*dt/dx = {cfl:.3f} (phải <1 để ổn định — bài học chính của Richardson 1922)")

for step in range(nsteps):
    dzeta_dx = (np.roll(zeta, -1) - np.roll(zeta, 1)) / (2 * dx)
    zeta = zeta - dt * U * dzeta_dx

print("Biên độ ban đầu:", round(float(np.max(np.abs(zeta0))), 4))
print("Biên độ sau", nsteps, "bước:", round(float(np.max(np.abs(zeta))), 4))
print("Ổn định số (biên độ không nổ) vì CFL < 1 — đây chính xác là vấn đề mà Richardson 1922 gặp phải")
print("khi chọn dt quá lớn, và Charney-Fjørtoft-von Neumann 1950 giải quyết bằng mô hình barotropic")
print("đơn giản hoá cùng bước thời gian hợp lý.")

dx=200000.0 m, dt=4000.0 s, CFL = U*dt/dx = 0.400 (phải <1 để ổn định — bài học chính của Richardson 1922)
Biên độ ban đầu: 1.0
Biên độ sau 400 bước: 1.3276
Ổn định số (biên độ không nổ) vì CFL < 1 — đây chính xác là vấn đề mà Richardson 1922 gặp phải
khi chọn dt quá lớn, và Charney-Fjørtoft-von Neumann 1950 giải quyết bằng mô hình barotropic
đơn giản hoá cùng bước thời gian hợp lý.


In [3]:
plt.figure(figsize=(7,3))
plt.plot(x/1000, zeta0, label='t=0')
plt.plot(x/1000, zeta, label=f't={nsteps*dt/3600:.1f} h')
plt.xlabel('Quãng đường theo vĩ tuyến (km)'); plt.ylabel('Xoáy tương đối (đơn vị tuỳ ý)')
plt.legend(); plt.title('Advection sóng Rossby — mô hình barotropic 1D đơn giản hoá')
plt.tight_layout(); plt.savefig('barotropic_advection.png', dpi=100)
print('Đã lưu barotropic_advection.png')

Đã lưu barotropic_advection.png
